In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import optuna


from typing import Callable
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, classification_report, RocCurveDisplay, PrecisionRecallDisplay, balanced_accuracy_score, brier_score_loss, roc_auc_score, f1_score, recall_score, make_scorer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from datetime import date
from enum import Enum
from typing import Callable

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=ConvergenceWarning) 
warnings.simplefilter(action='ignore', category=UserWarning)

NET BENEFIT AND WEIGHTED UTILITY

In [ ]:
def computeNetBenefit(y_true, y_proba, th=None):
  prop = len(y_true[y_true == 1])/len(y_true)
  y_pred = (y_proba >= th).astype(int)
  sens = recall_score(y_true, y_pred)
  spec = recall_score(y_true, y_pred, pos_label=0)
  return (sens*prop - (1-spec)*(1-prop)*th/(1-th))/prop


def computeWeightedUtility(y_true, y_proba, ths=None, relevances=None):
  if ths is None and relevances is None:
    return computeNetBenefit(y_true, y_proba)
  
  if np.isscalar(ths) and relevances is None:
    return computeNetBenefit(y_true, y_proba, ths)
  
  if relevances is None:
    relevances = np.ones(y_proba.shape)
  
  if ths is None:
    ths = np.ones(y_proba.shape)*0.5
  elif np.isscalar(ths):
    ths = np.ones(y_proba.shape)*ths

  if len(ths) != len(y_true):
    raise ValueError("If not scalar or None, ths should have the same length as y_true")
  if len(relevances) != len(y_true):
    raise ValueError("If not None, relevances should have the same length as y_true")

  pos_idx = y_true == 1
  rs = np.sum(relevances[pos_idx])
  pp = y_proba >= ths
  tp = np.logical_and(pos_idx, pp)
  fp = np.logical_and(np.logical_not(pos_idx), pp)
  return np.sum(tp*relevances)/rs - np.sum(ths/(1-ths)*fp*relevances)/rs

codice preso da mail Cabitza

In [ ]:
def snb_scorer(th, positiveLabel=1):
    # GridSearchCV accetta scoring(estimator, X, y) -> float
    def _score(estimator, X, y_true):
        if not hasattr(estimator, "predict_proba"):
            raise TypeError("Questo scorer richiede predict_proba().")
        proba = estimator.predict_proba(X)
        # colonna della classe positiva
        classes = list(estimator.classes_)
        y_proba = proba[:, classes.index(positiveLabel)]
        return computeNetBenefit(y_true, y_proba, th)
    return _score

In [ ]:
def snb_mean_over_thresholds_scorer(ths=(0.25, 0.5, 0.75), positiveLabel=1):
    ths = np.array(ths, dtype=float)

    def _score(estimator, X, y_true):
        if not hasattr(estimator, "predict_proba"):
            raise TypeError("Questo scorer richiede predict_proba().")
        proba = estimator.predict_proba(X)
        # colonna della classe positiva
        classes = list(estimator.classes_)
        y_proba = proba[:, classes.index(positiveLabel)]

        vals = [computeNetBenefit(y_true, y_proba, th) for th in ths]
        return float(np.mean(vals))
    return _score

scorer weighted utility

In [ ]:
def scorerWU(ths: np.ndarray, positiveLabel = 1):

    def _score(estimator, X, y_true):
        if not hasattr(estimator, "predict_proba"):
            raise TypeError("Questo scorer richiede predict_proba().")
        proba = estimator.predict_proba(X)
        # colonna della classe positiva
        classes = list(estimator.classes_)
        y_proba = proba[:, classes.index(positiveLabel)]
        return computeWeightedUtility(y_true, y_proba, ths)
    
    return _score

Se ogni riga del tuo dataset ha davvero la sua specifica soglia, scikit-learn non è in grado di "splittare" l'array ths automaticamente durante la cross-validazione, come fa con X e y.
Soluzione: Devi aggiungere ths come ultima colonna del tuo xTrain in modo che venga splittata insieme alle feature. Nello scorer, prima di passarlo al modello, estrai quella colonna

ML MODELS

In [ ]:
ODIdataframe  = pd.read_csv('ODIdataframe.csv')
COMIdataframe = pd.read_csv('COMIdataframe.csv')
SF36dataframe = pd.read_csv('SF36dataframe.csv')

In [ ]:
def prepareData(questionnaire: str):
    if questionnaire == "ODI":
        dataframe = ODIdataframe.copy()
        target = "overuse_PreOp_3months_ODI"
        risk = "overuseRiskODI"
    elif questionnaire == "COMI":
        dataframe = COMIdataframe.copy()
        target = "overuse_PreOp_3months_COMI"
        risk = "overuseRiskCOMI"
    elif questionnaire == "SF36":
        dataframe = SF36dataframe.copy()
        target = "overuse_PreOp_3months_SF36"
        risk = "overuseRiskSF36"
    
    dataframe = dataframe.dropna(subset=[target])
    features = [col for col in dataframe.columns if col != target]
    x = dataframe[features]
    y = dataframe[target]

    xTrain, xTest, yTrain, yTest = train_test_split(x, y, test_size=0.2, random_state=883)

    riskTest = xTest[risk]
    riskTrain = xTrain[risk]

    xTrain.drop(columns=[risk], inplace=True)
    xTest.drop(columns=[risk], inplace=True)

    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()
    
    xTrainScaled = scaler.fit_transform(imputer.fit_transform(xTrain))
    xTestScaled = scaler.transform(imputer.transform(xTest))
    
    return xTrainScaled, xTestScaled, yTrain, yTest, riskTest, riskTrain

In [ ]:
xTrainScaledODI, xTestScaledODI, yTrainODI, yTestODI, riskTestODI, riskTrainODI = prepareData("ODI")
xTrainScaledCOMI, xTestScaledCOMI, yTrainCOMI, yTestCOMI, riskTestCOMI, riskTrainCOMI = prepareData("COMI")
xTrainScaledSF36, xTestScaledSF36, yTrainSF36, yTestSF36, riskTestSF36, riskTrainSF36 = prepareData("SF36")

xTrain = {
    "ODI": xTrainScaledODI,
    "COMI": xTrainScaledCOMI,
    "SF36": xTrainScaledSF36
}

xTest = {
    "ODI": xTestScaledODI,
    "COMI": xTestScaledCOMI,
    "SF36": xTestScaledSF36
}

yTrain = {
    "ODI": yTrainODI,
    "COMI": yTrainCOMI,
    "SF36": yTrainSF36
}

yTest = {
    "ODI": yTestODI,
    "COMI": yTestCOMI,
    "SF36": yTestSF36
}

riskTest = {
    "ODI": riskTestODI,
    "COMI": riskTestCOMI,
    "SF36": riskTestSF36
}

riskTrain = {
    "ODI": riskTrainODI,
    "COMI": riskTrainCOMI,
    "SF36": riskTrainSF36
}

In [ ]:
def computeAdditionalMetrics(yTrue, yPred, yProb):
    balancedAccuracy = balanced_accuracy_score(yTrue, yPred)
    overallF1 = f1_score(yTrue, yPred, average='macro')
    brierScore = brier_score_loss(yTrue, yProb)
    aucScore = roc_auc_score(yTrue, yProb)

    return balancedAccuracy, overallF1, brierScore, aucScore

HYPERPARAMETER OPTIMIZATION

In [ ]:
def optimizeOptunaLR(questionnaire: str, scoringMetric: str|Callable):
    
    def optimizeLogisticRegression(trial):
            cValue = trial.suggest_float('C', 1e-3, 1e2, log=True)
            penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
            
            model = LogisticRegression(
                solver='liblinear',
                penalty=penalty,
                C=cValue,
                random_state=883
            )
            
            scores = cross_val_score(
                model, 
                xTrain[questionnaire], 
                yTrain[questionnaire], 
                cv=5, 
                scoring=scoringMetric, 
                n_jobs=-1
            )
            
            return scores.mean()

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(direction='maximize')
    study.optimize(optimizeLogisticRegression, n_trials=100) 
    print(f"Best parameters for {questionnaire}: {study.best_params}")
    return study

In [ ]:
def optimizeOptunaSVM(questionnaire: str, scoringMetric: str):

    def objective(trial):
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly'])
        cValue = trial.suggest_float('C', 0.1, 50.0, log=True)
            
        if kernel == 'linear':
            model = SVC(kernel=kernel, C=cValue, class_weight='balanced', random_state=42, cache_size=1000, probability=True)
            
        elif kernel == 'rbf':
            gamma = trial.suggest_categorical('gamma_rbf', ['scale', 'auto', 0.1, 0.01])
            model = SVC(kernel=kernel, C=cValue, gamma=gamma, class_weight='balanced', random_state=42, cache_size=1000, probability=True)
            
        elif kernel == 'poly':
            gamma = trial.suggest_categorical('gamma_poly', ['scale', 'auto'])
            degree = trial.suggest_int('degree', 2, 3)
            model = SVC(kernel=kernel, C=cValue, gamma=gamma, degree=degree, class_weight='balanced', random_state=42, cache_size=1000, probability=True)

        scores = cross_val_score(model, xTrain[questionnaire], yTrain[questionnaire], cv=5, scoring=scoringMetric, n_jobs=-1)
        return scores.mean()

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=80) 

    print(f"Best params: {study.best_params}\n")
    return study

In [ ]:
def optimizeOptunaXGB(questionnaire: str, scoringMetric: str):

    def objective(trial):
        neg_count = np.sum(yTrain[questionnaire] == 0)
        pos_count = np.sum(yTrain[questionnaire] == 1)
        weight = neg_count / pos_count if pos_count > 0 else 1
        
        params = {
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 7),
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'scale_pos_weight': weight,
            'eval_metric': 'logloss',
            'random_state': 42,
            'n_jobs': 1 
        }

        model = XGBClassifier(**params)
        scores = cross_val_score(model, xTrain[questionnaire], yTrain[questionnaire], cv=5, scoring=scoringMetric, n_jobs=-1)
        return scores.mean()

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=50)
    print(f"Best params: {study.best_params}\n")
    return study

In [ ]:
def optimizeOptunaRF(questionnaire: str, scoringMetric: str):

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 400),
            'max_depth': trial.suggest_categorical('max_depth', [None, 5, 10, 15, 20]),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
            'class_weight': 'balanced',
            'random_state': 42,
            'n_jobs': 1 
        }

        model = RandomForestClassifier(**params)
        
        scores = cross_val_score(model, xTrain[questionnaire], yTrain[questionnaire], cv=5, scoring=scoringMetric, n_jobs=-1)
        return scores.mean()
    
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=50)

    print(f"Best parameters: {study.best_params}\n")

    return study

In [ ]:
studyLRODI = optimizeOptunaLR("ODI", "balanced_accuracy")
studyLRCOMI = optimizeOptunaLR("COMI", "balanced_accuracy")
studyLRSF36 = optimizeOptunaLR("SF36", "balanced_accuracy")

studySVMODI = optimizeOptunaSVM("ODI", "balanced_accuracy")
studySVMCOMI = optimizeOptunaSVM("COMI", "balanced_accuracy")
studySVMSF36 = optimizeOptunaSVM("SF36", "balanced_accuracy")

studyXGBODI = optimizeOptunaXGB("ODI", "balanced_accuracy")
studyXGBCOMI = optimizeOptunaXGB("COMI", "balanced_accuracy")
studyXGBSF36 = optimizeOptunaXGB("SF36", "balanced_accuracy")

studyRFODI = optimizeOptunaRF("ODI", "balanced_accuracy")
studyRFCOMI = optimizeOptunaRF("COMI", "balanced_accuracy")
studyRFSF36 = optimizeOptunaRF("SF36", "balanced_accuracy")



studiesLR = {
    "ODI": studyLRODI,
    "COMI": studyLRCOMI,
    "SF36": studyLRSF36
}

studiesSVM = {
    "ODI": studySVMODI,
    "COMI": studySVMCOMI,
    "SF36": studySVMSF36
}

studiesXGB = {
    "ODI": studyXGBODI,
    "COMI": studyXGBCOMI,
    "SF36": studyXGBSF36
}

studiesRF = {
    "ODI": studyRFODI,
    "COMI": studyRFCOMI,
    "SF36": studyRFSF36
}

LOGISTIC REGRESSION

In [ ]:
def logisticRegression(questionnaire: str):
    study = studiesLR[questionnaire]

    logisticModel = LogisticRegression(
        solver='liblinear',
        random_state=883,
        **(study.best_params)
    )
    logisticModel.fit(xTrain[questionnaire], yTrain[questionnaire])

    prediction = logisticModel.predict(xTest[questionnaire])
    yProbs = logisticModel.predict_proba(xTest[questionnaire])[:, 1]

    return logisticModel, prediction, yProbs, \
           confusion_matrix(yTest[questionnaire], prediction), classification_report(yTest[questionnaire], prediction)

In [ ]:
def svm(questionnaire: str):
    study = studiesSVM[questionnaire]
    best_params = study.best_params
    
    final_svc_kwargs = {
        'kernel': best_params['kernel'], 
        'C': best_params['C'],
        'class_weight': 'balanced',
        'probability': True,
        'random_state': 42,
        'cache_size': 1000
    }
    
    if best_params['kernel'] == 'rbf':
        final_svc_kwargs['gamma'] = best_params['gamma_rbf']
    elif best_params['kernel'] == 'poly':
        final_svc_kwargs['gamma'] = best_params['gamma_poly']
        final_svc_kwargs['degree'] = best_params['degree']

    bestModel = SVC(**final_svc_kwargs)
    bestModel.fit(xTrain[questionnaire], yTrain[questionnaire])

    prediction = bestModel.predict(xTest[questionnaire])
    yProb = bestModel.predict_proba(xTest[questionnaire])[:, 1]

    return bestModel, prediction, yProb, \
           confusion_matrix(yTest[questionnaire], prediction), classification_report(yTest[questionnaire], prediction)

In [ ]:
def xgboostModel(questionnaire: str):
    study = studiesXGB[questionnaire]
    neg_count = np.sum(yTrain[questionnaire] == 0)
    pos_count = np.sum(yTrain[questionnaire] == 1)
    weight = neg_count / pos_count if pos_count > 0 else 1

    bestModel = XGBClassifier(
        scale_pos_weight=weight,
        eval_metric='logloss',
        random_state=42,
        **study.best_params
    )
    bestModel.fit(xTrain[questionnaire], yTrain[questionnaire])

    prediction = bestModel.predict(xTest[questionnaire])
    yProb = bestModel.predict_proba(xTest[questionnaire])[:, 1]

    return bestModel, prediction, yProb, \
           confusion_matrix(yTest[questionnaire], prediction), classification_report(yTest[questionnaire], prediction)

In [ ]:
def randomForestModel(questionnaire: str):
    study = studiesRF[questionnaire]

    bestModel = RandomForestClassifier(
        **study.best_params,
        class_weight='balanced',
        random_state=42
    )
    bestModel.fit(xTrain[questionnaire], yTrain[questionnaire])

    prediction = bestModel.predict(xTest[questionnaire])
    yProb = bestModel.predict_proba(xTest[questionnaire])[:, 1]

    return bestModel, prediction, yProb, \
           confusion_matrix(yTest[questionnaire], prediction), classification_report(yTest[questionnaire], prediction)

In [ ]:
class FittedClassifier:
    def __init__(self, modelName: str, questionnaire: str):
        match modelName:
            case "Logistic Regression": model = logisticRegression
            case "SVM": model = svm
            case "XG Boost": model = xgboostModel
            case "Random Forest": model = randomForestModel
        
        self.model, self.prediction, self.yProbs, self.confusionMatrix, self.report = model(questionnaire)

In [ ]:
logisticModels = {
    "ODI": FittedClassifier("Logistic Regression", "ODI"),
    "COMI": FittedClassifier("Logistic Regression", "COMI"),
    "SF36": FittedClassifier("Logistic Regression", "SF36"),
}

svmModels = {
    "ODI": FittedClassifier("SVM", "ODI"),
    "COMI": FittedClassifier("SVM", "COMI"),
    "SF36": FittedClassifier("SVM", "SF36"),
}

xgboostModels = {
    "ODI": FittedClassifier("XG Boost", "ODI"),
    "COMI": FittedClassifier("XG Boost", "COMI"),
    "SF36": FittedClassifier("XG Boost", "SF36"),
}

randomForestModels = {
    "ODI": FittedClassifier("Random Forest", "ODI"),
    "COMI": FittedClassifier("Random Forest", "COMI"),
    "SF36": FittedClassifier("Random Forest", "SF36"),
}

EVALUATION

In [ ]:
def evaluateModelNBWU(questionnaire: str, model: FittedClassifier):
    global yTest, xTest, riskTest

    netBenefit25 = computeNetBenefit(yTest[questionnaire], model.yProbs, th=0.25)
    netBenefit50 = computeNetBenefit(yTest[questionnaire], model.yProbs, th=0.50)
    netBenefit75 = computeNetBenefit(yTest[questionnaire], model.yProbs, th=0.75)

    netBenefitMedio = snb_mean_over_thresholds_scorer()(model.model, xTest[questionnaire], yTest[questionnaire])
    weightedUtility = computeWeightedUtility(yTest[questionnaire], model.yProbs, riskTest[questionnaire])

    print(f"Net Benefit 0.25: {netBenefit25:.4f}")
    print(f"Net Benefit 0.50: {netBenefit50:.4f}")
    print(f"Net Benefit 0.75: {netBenefit75:.4f}")
    print(f"Average Net Benefit: {netBenefitMedio:.4f}")
    print(f"Weighted utility: {weightedUtility}")

In [ ]:
def evaluateModel(questionnaire:str, model: dict[str, FittedClassifier]):
    model = model[questionnaire]
    fig, ax = plt.subplots(1, 2, figsize=(16, 6))

    sns.heatmap(model.confusionMatrix, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax[0])
    ax[0].set_xlabel('Predicted Label')
    ax[0].set_ylabel('True Label')
    ax[0].set_title(f'Confusion Matrix ({questionnaire})', fontsize=14)

    RocCurveDisplay.from_estimator(
        model.model, 
        xTest[questionnaire], 
        yTest[questionnaire], 
        name=model.model.__class__.__name__,
        ax=ax[1],
        curve_kwargs={'color': 'forestgreen'}
    )
    ax[1].plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)")
    ax[1].set_title(f"ROC Curve: Predicting Overuse ({questionnaire})", fontsize=14)
    ax[1].legend(loc='lower right') 

    plt.tight_layout() 
    plt.show()

    print("\n--- Classification Report ---")
    print(model.report)

    balancedAccuracy, overallF1, brierScore, aucScore = computeAdditionalMetrics(yTest[questionnaire], model.prediction, model.yProbs)
    print(f"Balanced Accuracy: {balancedAccuracy:.4f}")
    print(f"Overall F1 Score: {overallF1:.4f}")
    print(f"Brier Score: {brierScore:.4f}")
    print(f"AUC Score: {aucScore:.4f}")
    evaluateModelNBWU(questionnaire, model)

In [ ]:
evaluateModel("ODI", logisticModels)
evaluateModel("COMI", logisticModels)
evaluateModel("SF36", logisticModels)

svm

In [ ]:
evaluateModel("ODI", svmModels)
evaluateModel("COMI", svmModels)
evaluateModel("SF36", svmModels)

xgboost

In [ ]:
evaluateModel("ODI", xgboostModels)
evaluateModel("COMI", xgboostModels)
evaluateModel("SF36", xgboostModels)

random forest

In [ ]:
evaluateModel("ODI", randomForestModels)
evaluateModel("COMI", randomForestModels)
evaluateModel("SF36", randomForestModels)

totale di 6 modelli, 1 normale, 3 net benefit, 1 WU uno media risultati tau

modello net benefit -> un solo ths 
modello wu ths individuale -> interpolazione lineare (vai a cercare la formula)
valutazione

qua interpolazione lineare perchè così per ogni istanza ho un threshold diverso
se ths sempre uguale allora net benefit, se diverso (passi un vettore) allora WU
net benefit 4 valori (soglia fissa 0.25, 0.5, 0.75, media delle soglie)
vogliamo osservare (si spera) ottimizzare per WU da risultati miglior, non si sa per net benefit, meglio se relazione d'ordine balAcc < net ben < WU
aggiungi come metriche WU e netB

LOGISTIC CON NET BENEFIT

In [ ]:
# TODO: sistema codice 1. risk nei modelli 2. ordine del codice 3. feature selection

copia della cella con gli studi dove lo scorer per determinare i migliori iperparametri con optuna è il NB
per ciascun modello e questionario utilizzo come th del NB quello migliore a seconda dei risultati dell evaluation

svm ottimizzato con nb th = 25%

In [ ]:
svmStudies = {
    "ODI": optimizeOptunaSVM("ODI", snb_scorer(0.25))
}
svmODINB25 = FittedClassifier("SVM", "ODI")

In [ ]:
evaluateModel("ODI", {"ODI": svmODINB25})

In [ ]:
svmStudies = {
    "ODI": optimizeOptunaSVM("ODI", scorerWU(riskTrainODI))
}
svmODIWU = FittedClassifier("SVM", "ODI")

evaluateModel("ODI", {"ODI": svmODIWU})